<a href="https://colab.research.google.com/github/sw030701-ai/motor-control-optimization/blob/main/experiments/05_robustness_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. Robustness Test


## 목적 / 공정 비교 원칙

이 notebook의 목적은 nominal condition에서 확정된 세 controller가 motor parameter variation, external load disturbance, sensor noise에 대해 얼마나 안정적으로 성능을 유지하는지 확인하는 것이다.

비교 대상 controller는 다음 세 가지이다.

| Controller | 이 실험에서의 의미 |
|---|---|
| Manual PID | `02_pid_baseline_tuning`에서 확정한 manual baseline gain |
| Optimized PID | main branch의 공식 constrained PID optimization 결과 |
| RL Direct Voltage Control | best deterministic evaluation checkpoint의 TD3 actor |

공정 비교를 위해 robustness test 중에는 아래 작업을 하지 않는다.

- PID gain 재튜닝
- Bayesian optimizer 또는 random search 재실행
- RL policy 재학습

즉, nominal condition에서 이미 정해진 controller를 고정한 뒤, 동일한 test condition을 세 controller에 적용한다.


In [1]:
from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULT_TABLE_DIR = PROJECT_ROOT / "results" / "tables"
RESULT_FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
RESULT_MODEL_DIR = PROJECT_ROOT / "results" / "models"
RESULT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 160,
    "axes.grid": True,
    "grid.alpha": 0.3,
})


## 모터 / 컨트롤러 불러오기

Motor parameter는 논문 Table 1 기반 nominal model을 사용한다. 이 프로젝트에서는 논문 parameter 중 $J$를 rotor inertia $J_m$, $b$를 viscous friction coefficient로 사용한다.

기준 속도와 전압 제한은 기존 baseline tuning 결과에서 읽는다.

$$
\omega_{\mathrm{ref}} = 12.6\ \mathrm{rad/s},
\qquad
V_{\max}=12\ \mathrm{V}
$$

Manual PID와 Optimized PID도 main branch의 현재 저장 결과에서 읽는다. RL controller는 `results/models/td3_direct_voltage_best_actor.pt`에 저장된 best deterministic evaluation checkpoint만 사용한다.


In [2]:
from dataclasses import replace

from src.controller.pid import PIDController, PIDGains
from src.motor.dc_motor import DCMotor, DCMotorParams, NOMINAL_MOTOR_SOURCE, nominal_dc_motor_params
from src.optimization.cost_function import compute_cost
from src.rl.direct_voltage_env import DirectVoltageEnvConfig, DirectVoltageMotorEnv
from src.rl.td3 import TD3Agent

BASELINE_RECORD_PATH = RESULT_TABLE_DIR / "baseline_pid_tuning_record.csv"
OPTIMIZED_RECORD_PATH = RESULT_TABLE_DIR / "constrained_pid_optimization_summary.csv"
BEST_ACTOR_PATH = RESULT_MODEL_DIR / "td3_direct_voltage_best_actor.pt"


def load_fixed_conditions(params):
    if BASELINE_RECORD_PATH.exists():
        baseline = pd.read_csv(BASELINE_RECORD_PATH).iloc[0]
        return float(baseline["reference_speed_rad_s"]), float(baseline["V_max"])
    return 12.6, 12.0


def load_manual_gains():
    row = pd.read_csv(BASELINE_RECORD_PATH).iloc[0]
    return PIDGains(
        K_p=float(row["K_p_baseline"]),
        K_i=float(row["K_i_baseline"]),
        K_d=float(row["K_d_baseline"]),
    )


def load_optimized_gains():
    table = pd.read_csv(OPTIMIZED_RECORD_PATH)
    optimized = table[table["Controller"].isin(["Constrained Classical", "Optimized PID"])]
    if optimized.empty:
        raise ValueError("Optimized PID row was not found in constrained_pid_optimization_summary.csv")
    row = optimized.iloc[0]
    return PIDGains(
        K_p=float(row["K_p"]),
        K_i=float(row["K_i"]),
        K_d=float(row["K_d"]),
    )


params_nominal = nominal_dc_motor_params()
omega_ref, V_max = load_fixed_conditions(params_nominal)
manual_gains = load_manual_gains()
optimized_gains = load_optimized_gains()
rl_agent = TD3Agent.load_actor(BEST_ACTOR_PATH)
rl_metadata = TD3Agent.load_actor_metadata(BEST_ACTOR_PATH)

controller_info = pd.DataFrame([
    {"Controller": "Manual PID", **manual_gains.as_dict(), "Source": "baseline_pid_tuning_record.csv"},
    {"Controller": "Optimized PID", **optimized_gains.as_dict(), "Source": "constrained_pid_optimization_summary.csv"},
    {
        "Controller": "RL Direct Voltage Control",
        "K_p": np.nan,
        "K_i": np.nan,
        "K_d": np.nan,
        "Source": "td3_direct_voltage_best_actor.pt",
    },
])

print(f"omega_ref = {omega_ref:.4f} rad/s, V_max = {V_max:.1f} V")
display(controller_info)
print(json.dumps({
    "nominal_motor_source": NOMINAL_MOTOR_SOURCE,
    "rl_best_checkpoint": rl_metadata,
}, indent=2, ensure_ascii=False, default=str))


omega_ref = 12.6000 rad/s, V_max = 12.0 V


,Controller,K_p,K_i,K_d,Source
0,Manual PID,0.800000,2.000000,0.002000,baseline_pid_tuning_record.csv
1,Optimized PID,0.215097,0.996516,0.000855,constrained_pid_optimization_summary.csv
2,RL Direct Voltage Control,NaN,NaN,NaN,td3_direct_voltage_best_actor.pt


{
  "nominal_motor_source": {
    "title": "Enhanced DC motor speed regulation using two-degree-of-freedom PID controller tuned by animated oat optimization with simulation and real-time experimental validation",
    "doi": "10.1177/00202940261442256",
    "url": "https://journals.sagepub.com/doi/10.1177/00202940261442256",
    "table": "Table 1. DC motor parameters",
    "note": "The paper reports J, b, K_e, K_t, R, and L. This project maps J to J_m and b to viscous friction."
  },
  "rl_best_checkpoint": {
    "checkpoint_episode": 20,
    "checkpoint_metric": "J_total",
    "checkpoint_score": 0.03490333789728869,
    "actor_lr": 0.0001,
    "critic_lr": 0.0001,
    "selection_rule": "lowest deterministic evaluation J_total",
    "evaluation": {
      "J_total": 0.03490333789728869,
      "J_tracking": 0.007779457009577775,
      "J_overshoot": 0.0,
      "J_control": 0.20157109127694683,
      "overshoot_percent": 0.0,
      "settling_time": Infinity,
      "steady_state_error_perc

## Robustness 조건

### 1. Armature resistance variation

Nominal armature resistance는 다음과 같다.

$$
R_0 = 0.18644\ \Omega
$$

Resistance robustness test에서는 controller는 고정하고 motor model의 $R$만 바꾼다.

$$
R \in \{0.8R_0,\ R_0,\ 1.2R_0\}
  = \{0.149152,\ 0.18644,\ 0.223728\}\ \Omega
$$

### 2. Rotor inertia variation

Nominal rotor inertia는 다음과 같다.

$$
J_0 = 0.013767\ \mathrm{kg\,m^2}
$$

Inertia robustness test에서는 controller는 고정하고 motor model의 $J_m$만 바꾼다.

$$
J_m \in \{0.8J_0,\ J_0,\ 1.2J_0\}
    = \{0.0110136,\ 0.013767,\ 0.0165204\}\ \mathrm{kg\,m^2}
$$

### 3. Load torque disturbance

Motor equation에는 이미 viscous friction torque $b\omega$가 포함되어 있다.

$$
J_m\frac{d\omega}{dt}
= K_t i - b\omega - T_L
$$

따라서 $b\omega$를 external load $T_L$로 다시 넣으면 내부 마찰과 외부 부하를 중복 계산하게 된다. 이 실험에서는 nominal external load를 0으로 두고, 5초부터 project-specific sudden external load만 추가한다.

$$
\Delta T_L
= 0.25\,b\,\omega_{\mathrm{ref}}
=0.25 \times 0.049813 \times 12.6
\approx 0.1569\ \mathrm{N\,m}
$$

$$
T_L(t)=
\begin{cases}
0, & 0 \le t < 5\ \mathrm{s} \\
0.1569\ \mathrm{N\,m}, & 5 \le t \le 10\ \mathrm{s}
\end{cases}
$$

논문은 sudden 25% load increase라는 조건을 참고할 수 있게 제공하지만, 위의 $0.1569\ \mathrm{N\,m}$ 절대값은 논문에 직접 제시된 값이 아니다. 이 값은 이 프로젝트의 nominal friction torque $b\omega_{\mathrm{ref}}$의 25%로 환산한 project-specific robustness condition이다.

### 4. Sensor noise robustness

Sensor noise는 motor의 true state가 아니라 controller가 보는 speed feedback measurement에만 적용한다.

$$
\omega_{\mathrm{measured}}(t)
= \omega_{\mathrm{true}}(t) + n(t),
\qquad
n(t)\sim\mathcal{N}(0,\sigma^2)
$$

$$
\sigma \in \{0,\ 0.005,\ 0.01\}\ \mathrm{rad/s}
$$

Current 측정 noise는 추가하지 않는다. 실험 정의를 단순하고 재현 가능하게 유지하기 위해 speed feedback noise만 사용한다. 확률적 실험이므로 각 $\sigma$마다 5개 seed를 사용하고 평균과 표준편차를 함께 기록한다.


In [3]:
R0 = params_nominal.R
J0 = params_nominal.J_m
b = params_nominal.b
LOAD_STEP_TIME = 5.0
LOAD_STEP_VALUE = round(0.25 * b * omega_ref, 4)
SIMULATION_TIME = 10.0
DT = 0.001
NOISE_SEEDS = [101, 102, 103, 104, 105]

robustness_conditions = {
    "resistance": [
        {"condition": "R_0.8x", "R": 0.8 * R0, "description": "R = 0.8 R0"},
        {"condition": "R_1.0x", "R": R0, "description": "R = R0"},
        {"condition": "R_1.2x", "R": 1.2 * R0, "description": "R = 1.2 R0"},
    ],
    "inertia": [
        {"condition": "J_0.8x", "J_m": 0.8 * J0, "description": "J_m = 0.8 J0"},
        {"condition": "J_1.0x", "J_m": J0, "description": "J_m = J0"},
        {"condition": "J_1.2x", "J_m": 1.2 * J0, "description": "J_m = 1.2 J0"},
    ],
    "load_disturbance": [
        {
            "condition": "load_step_0.1569_Nm",
            "load_step_time": LOAD_STEP_TIME,
            "load_step_value": LOAD_STEP_VALUE,
            "description": "T_L = 0 before 5 s, then 0.1569 N m",
        },
    ],
    "sensor_noise": [
        {"condition": "noise_sigma_0.000", "sigma": 0.0, "description": "sigma = 0 rad/s"},
        {"condition": "noise_sigma_0.005", "sigma": 0.005, "description": "sigma = 0.005 rad/s"},
        {"condition": "noise_sigma_0.010", "sigma": 0.01, "description": "sigma = 0.01 rad/s"},
    ],
}

condition_table = pd.DataFrame([
    {"Test": "Resistance", "Condition": "0.8 R0", "Value": f"{0.8 * R0:.6f} ohm"},
    {"Test": "Resistance", "Condition": "R0", "Value": f"{R0:.6f} ohm"},
    {"Test": "Resistance", "Condition": "1.2 R0", "Value": f"{1.2 * R0:.6f} ohm"},
    {"Test": "Inertia", "Condition": "0.8 J0", "Value": f"{0.8 * J0:.7f} kg m^2"},
    {"Test": "Inertia", "Condition": "J0", "Value": f"{J0:.6f} kg m^2"},
    {"Test": "Inertia", "Condition": "1.2 J0", "Value": f"{1.2 * J0:.7f} kg m^2"},
    {"Test": "Load disturbance", "Condition": "5 s step", "Value": f"{LOAD_STEP_VALUE:.4f} N m"},
    {"Test": "Sensor noise", "Condition": "sigma", "Value": "{0, 0.005, 0.01} rad/s"},
])
display(condition_table)


,Test,Condition,Value
0,Resistance,0.8 R0,0.149152 ohm
1,Resistance,R0,0.186440 ohm
2,Resistance,1.2 R0,0.223728 ohm
3,Inertia,0.8 J0,0.0110136 kg m^2
4,Inertia,J0,0.013767 kg m^2
5,Inertia,1.2 J0,0.0165204 kg m^2
6,Load disturbance,5 s step,0.1569 N m
7,Sensor noise,sigma,"{0, 0.005, 0.01} rad/s"


## 공통 평가 함수

세 controller 모두 동일한 motor model과 동일한 metric으로 평가한다.

기존 프로젝트의 cost function을 그대로 사용한다.

$$
J
=0.60J_{\mathrm{tracking}}
+0.25J_{\mathrm{overshoot}}
+0.15J_{\mathrm{control}}
$$

결과표에는 다음 metric을 저장한다.

| Metric | 의미 |
|---|---|
| `J_total` | weighted total cost |
| `J_tracking` | time-weighted normalized tracking error |
| `J_overshoot` | overshoot penalty |
| `J_control` | normalized voltage effort |
| `overshoot`, `overshoot_percent` | peak speed 기준 overshoot |
| `settling_time` | 전체 episode 기준 $\pm2\%$ band settling time |
| `steady_state_error_percent` | 마지막 10% 구간 평균 기준 steady-state error |
| `voltage_max_abs` | 최대 절대 전압 |
| `saturation_percent` | $\lvert V\rvert\ge0.99V_{\max}$ 비율 |
| `omega_final` | final speed |

Load disturbance는 전체 episode 기준 `settling_time`만으로는 5초 이후 외란 복구 성능을 잘 표현하지 못한다. 그래서 별도 metric인 disturbance recovery time을 추가한다.

$$
t_{\mathrm{recovery}}
=
\min_{\tau \ge t_d}
\left\{
\tau-t_d\ \middle|\ 
\left|\omega(t)-\omega_{\mathrm{ref}}\right|
\le 0.02|\omega_{\mathrm{ref}}|,
\ \forall t\in[\tau,T]
\right\}
$$

여기서 $t_d=5\ \mathrm{s}$이고 $T=10\ \mathrm{s}$이다. 외란 이후 끝까지 $\pm2\%$ band 안에 유지되는 시점이 없으면 $t_{\mathrm{recovery}}=\infty$로 기록한다.


In [4]:
def make_params(base_params, **updates):
    return replace(base_params, **updates)


def load_torque_at(time_value, step_time=None, step_value=0.0):
    if step_time is None:
        return 0.0
    return float(step_value) if time_value >= step_time else 0.0


def time_vector(simulation_time=SIMULATION_TIME, dt=DT):
    return np.arange(0.0, simulation_time + 0.5 * dt, dt)


def simulate_pid_fixed_controller(
    motor_params,
    gains,
    omega_ref=omega_ref,
    V_max=V_max,
    simulation_time=SIMULATION_TIME,
    dt=DT,
    load_step_time=None,
    load_step_value=0.0,
    sensor_noise_sigma=0.0,
    seed=None,
):
    motor = DCMotor(motor_params)
    controller = PIDController.from_gains(gains, output_min=-V_max, output_max=V_max)
    rng = np.random.default_rng(seed)
    time = time_vector(simulation_time, dt)

    omega_history = np.zeros_like(time)
    measured_history = np.zeros_like(time)
    current_history = np.zeros_like(time)
    voltage_history = np.zeros_like(time)
    error_history = np.zeros_like(time)
    load_history = np.zeros_like(time)

    for k, t in enumerate(time):
        noise = 0.0 if sensor_noise_sigma == 0.0 else rng.normal(0.0, sensor_noise_sigma)
        omega_measured = motor.omega + noise
        voltage, error = controller.compute(omega_ref, omega_measured, dt)
        load_torque = load_torque_at(t, load_step_time, load_step_value)
        current, omega = motor.step(voltage=voltage, dt=dt, load_torque=load_torque)

        omega_history[k] = omega
        measured_history[k] = omega_measured
        current_history[k] = current
        voltage_history[k] = voltage
        error_history[k] = error
        load_history[k] = load_torque

    return {
        "time": time,
        "omega": omega_history,
        "omega_measured": measured_history,
        "current": current_history,
        "voltage": voltage_history,
        "error": error_history,
        "load_torque": load_history,
        "omega_ref": omega_ref,
        "V_max": V_max,
    }


def simulate_rl_fixed_controller(
    motor_params,
    agent,
    omega_ref=omega_ref,
    V_max=V_max,
    simulation_time=SIMULATION_TIME,
    dt=DT,
    load_step_time=None,
    load_step_value=0.0,
    sensor_noise_sigma=0.0,
    seed=None,
):
    config = DirectVoltageEnvConfig(
        omega_ref=omega_ref,
        V_max=V_max,
        simulation_time=simulation_time,
        dt=dt,
    )
    helper_env = DirectVoltageMotorEnv(motor_params, config=config)
    motor = DCMotor(motor_params)
    rng = np.random.default_rng(seed)
    time = time_vector(simulation_time, dt)

    omega_history = np.zeros_like(time)
    measured_history = np.zeros_like(time)
    current_history = np.zeros_like(time)
    voltage_history = np.zeros_like(time)
    error_history = np.zeros_like(time)
    load_history = np.zeros_like(time)

    for k, t in enumerate(time):
        noise = 0.0 if sensor_noise_sigma == 0.0 else rng.normal(0.0, sensor_noise_sigma)
        omega_measured = motor.omega + noise
        observation_for_controller = np.array(
            [omega_ref - omega_measured, omega_measured, motor.current],
            dtype=float,
        )
        normalized_observation = helper_env.normalize_observation(observation_for_controller)
        voltage = float(agent.select_action(normalized_observation)[0])
        voltage = float(np.clip(voltage, -V_max, V_max))
        load_torque = load_torque_at(t, load_step_time, load_step_value)
        current, omega = motor.step(voltage=voltage, dt=dt, load_torque=load_torque)

        omega_history[k] = omega
        measured_history[k] = omega_measured
        current_history[k] = current
        voltage_history[k] = voltage
        error_history[k] = omega_ref - omega_measured
        load_history[k] = load_torque

    return {
        "time": time,
        "omega": omega_history,
        "omega_measured": measured_history,
        "current": current_history,
        "voltage": voltage_history,
        "error": error_history,
        "load_torque": load_history,
        "omega_ref": omega_ref,
        "V_max": V_max,
    }


def disturbance_recovery_time(result, omega_ref=omega_ref, disturbance_time=LOAD_STEP_TIME, tolerance=0.02):
    time = np.asarray(result["time"], dtype=float)
    omega = np.asarray(result["omega"], dtype=float)
    band = tolerance * abs(omega_ref)
    after_disturbance = np.where(time >= disturbance_time)[0]
    if len(after_disturbance) == 0:
        return np.nan

    error = np.abs(omega - omega_ref)
    for idx in after_disturbance:
        if np.all(error[idx:] <= band):
            return float(time[idx] - disturbance_time)
    return np.inf


def metric_row(controller_name, test_group, condition_name, condition_label, result, seed=np.nan, **details):
    cost = compute_cost(result, omega_ref=omega_ref, V_max=V_max)
    row = {
        "Controller": controller_name,
        "test_group": test_group,
        "condition": condition_name,
        "condition_label": condition_label,
        "seed": seed,
        "J_total": cost["total"],
        "J_tracking": cost["tracking"],
        "J_overshoot": cost["overshoot_cost"],
        "J_control": cost["control"],
        "overshoot": cost["overshoot"],
        "overshoot_percent": cost["overshoot_percent"],
        "settling_time": cost["settling_time"],
        "steady_state_error_percent": cost["steady_state_error_percent"],
        "voltage_max_abs": cost["voltage_max_abs"],
        "saturation_percent": cost["saturation_percent"],
        "omega_final": cost["omega_final"],
        "disturbance_recovery_time": (
            disturbance_recovery_time(result) if test_group == "load_disturbance" else np.nan
        ),
    }
    row.update(details)
    return row


controllers = [
    {"name": "Manual PID", "type": "pid", "gains": manual_gains},
    {"name": "Optimized PID", "type": "pid", "gains": optimized_gains},
    {"name": "RL Direct Voltage Control", "type": "rl", "agent": rl_agent},
]


def simulate_controller(controller, motor_params, **kwargs):
    if controller["type"] == "pid":
        return simulate_pid_fixed_controller(motor_params, controller["gains"], **kwargs)
    if controller["type"] == "rl":
        return simulate_rl_fixed_controller(motor_params, controller["agent"], **kwargs)
    raise ValueError(f"Unknown controller type: {controller['type']}")


## 각 실험 실행

아래 cell은 네 종류의 robustness test를 모두 실행하고, 개별 run 결과를 하나의 CSV로 저장한다.

Sensor noise는 각 $\sigma$마다 5회 반복한다.


In [5]:
all_rows = []
representative_results = {}

# Nominal 기준 run: degradation 계산의 기준점
for controller in controllers:
    result = simulate_controller(controller, params_nominal)
    row = metric_row(
        controller["name"],
        "nominal",
        "nominal",
        "Nominal",
        result,
        R=params_nominal.R,
        J_m=params_nominal.J_m,
        load_step_value=0.0,
        sensor_noise_sigma=0.0,
    )
    all_rows.append(row)
    representative_results[(controller["name"], "nominal", "nominal")] = result

# 1) Armature resistance variation
for condition in robustness_conditions["resistance"]:
    params = make_params(params_nominal, R=condition["R"])
    for controller in controllers:
        result = simulate_controller(controller, params)
        all_rows.append(metric_row(
            controller["name"],
            "resistance",
            condition["condition"],
            condition["description"],
            result,
            R=params.R,
            J_m=params.J_m,
            load_step_value=0.0,
            sensor_noise_sigma=0.0,
        ))
        representative_results[(controller["name"], "resistance", condition["condition"])] = result

# 2) Rotor inertia variation
for condition in robustness_conditions["inertia"]:
    params = make_params(params_nominal, J_m=condition["J_m"])
    for controller in controllers:
        result = simulate_controller(controller, params)
        all_rows.append(metric_row(
            controller["name"],
            "inertia",
            condition["condition"],
            condition["description"],
            result,
            R=params.R,
            J_m=params.J_m,
            load_step_value=0.0,
            sensor_noise_sigma=0.0,
        ))
        representative_results[(controller["name"], "inertia", condition["condition"])] = result

# 3) Load torque disturbance
for condition in robustness_conditions["load_disturbance"]:
    for controller in controllers:
        result = simulate_controller(
            controller,
            params_nominal,
            load_step_time=condition["load_step_time"],
            load_step_value=condition["load_step_value"],
        )
        all_rows.append(metric_row(
            controller["name"],
            "load_disturbance",
            condition["condition"],
            condition["description"],
            result,
            R=params_nominal.R,
            J_m=params_nominal.J_m,
            load_step_time=condition["load_step_time"],
            load_step_value=condition["load_step_value"],
            sensor_noise_sigma=0.0,
        ))
        representative_results[(controller["name"], "load_disturbance", condition["condition"])] = result

# 4) Sensor noise robustness
for condition in robustness_conditions["sensor_noise"]:
    for seed in NOISE_SEEDS:
        for controller in controllers:
            result = simulate_controller(
                controller,
                params_nominal,
                sensor_noise_sigma=condition["sigma"],
                seed=seed,
            )
            all_rows.append(metric_row(
                controller["name"],
                "sensor_noise",
                condition["condition"],
                condition["description"],
                result,
                seed=seed,
                R=params_nominal.R,
                J_m=params_nominal.J_m,
                load_step_value=0.0,
                sensor_noise_sigma=condition["sigma"],
            ))
            if seed == NOISE_SEEDS[0]:
                representative_results[(controller["name"], "sensor_noise", condition["condition"])] = result

raw_results = pd.DataFrame(all_rows)

nominal_cost = (
    raw_results[raw_results["test_group"] == "nominal"]
    .set_index("Controller")["J_total"]
)
raw_results["J_nominal"] = raw_results["Controller"].map(nominal_cost)
raw_results["performance_degradation_percent"] = (
    (raw_results["J_total"] - raw_results["J_nominal"])
    / raw_results["J_nominal"]
    * 100.0
)

raw_results_path = RESULT_TABLE_DIR / "robustness_all_results.csv"
raw_results.to_csv(raw_results_path, index=False)

print(f"Saved: {raw_results_path.relative_to(PROJECT_ROOT)}")
display(raw_results.head(12))


Saved: results/tables/robustness_all_results.csv


,Controller,test_group,condition,condition_label,seed,J_total,J_tracking,J_overshoot,J_control,overshoot,...,saturation_percent,omega_final,disturbance_recovery_time,R,J_m,load_step_value,sensor_noise_sigma,load_step_time,J_nominal,performance_degradation_percent
0,Manual PID,nominal,nominal,Nominal,NaN,0.038177,0.000114,0.000000e+00,0.254054,0.000000e+00,...,0.000000,12.600000,NaN,0.186440,0.013767,0.0,0.0,NaN,0.038177,0.000000
1,Optimized PID,nominal,nominal,Nominal,NaN,0.036702,0.000611,7.499755e-09,0.242238,8.660113e-05,...,0.000000,12.600000,NaN,0.186440,0.013767,0.0,0.0,NaN,0.036702,0.000000
2,RL Direct Voltage Control,nominal,nominal,Nominal,NaN,0.034903,0.007779,0.000000e+00,0.201571,0.000000e+00,...,0.599940,11.033782,NaN,0.186440,0.013767,0.0,0.0,NaN,0.034903,0.000000
3,Manual PID,resistance,R_0.8x,R = 0.8 R0,NaN,0.025356,0.000073,0.000000e+00,0.168746,0.000000e+00,...,0.000000,12.600000,NaN,0.149152,0.013767,0.0,0.0,NaN,0.038177,-33.582431
4,Optimized PID,resistance,R_0.8x,R = 0.8 R0,NaN,0.024402,0.000412,2.111682e-06,0.161025,1.453163e-03,...,0.000000,12.600000,NaN,0.149152,0.013767,0.0,0.0,NaN,0.036702,-33.514684
5,RL Direct Voltage Control,resistance,R_0.8x,R = 0.8 R0,NaN,0.024389,0.005754,0.000000e+00,0.139577,0.000000e+00,...,0.589941,11.252654,NaN,0.149152,0.013767,0.0,0.0,NaN,0.034903,-30.124211
6,Manual PID,resistance,R_1.0x,R = R0,NaN,0.038177,0.000114,0.000000e+00,0.254054,0.000000e+00,...,0.000000,12.600000,NaN,0.186440,0.013767,0.0,0.0,NaN,0.038177,0.000000
7,Optimized PID,resistance,R_1.0x,R = R0,NaN,0.036702,0.000611,7.499755e-09,0.242238,8.660113e-05,...,0.000000,12.600000,NaN,0.186440,0.013767,0.0,0.0,NaN,0.036702,0.000000
8,RL Direct Voltage Control,resistance,R_1.0x,R = R0,NaN,0.034903,0.007779,0.000000e+00,0.201571,0.000000e+00,...,0.599940,11.033782,NaN,0.186440,0.013767,0.0,0.0,NaN,0.034903,0.000000
9,Manual PID,resistance,R_1.2x,R = 1.2 R0,NaN,0.053532,0.000166,0.000000e+00,0.356213,0.000000e+00,...,0.000000,12.600000,NaN,0.223728,0.013767,0.0,0.0,NaN,0.038177,40.221658


## Controller별 결과표

Sensor noise는 seed 반복이 있으므로 평균과 표준편차를 함께 요약한다. 나머지 deterministic test는 표준편차가 0 또는 `NaN`에 가깝다.


In [6]:
metric_columns = [
    "J_total",
    "J_tracking",
    "J_overshoot",
    "J_control",
    "overshoot_percent",
    "settling_time",
    "steady_state_error_percent",
    "voltage_max_abs",
    "saturation_percent",
    "omega_final",
    "disturbance_recovery_time",
    "performance_degradation_percent",
]

summary_mean = (
    raw_results
    .groupby(["test_group", "condition", "condition_label", "Controller"], dropna=False)[metric_columns]
    .mean()
    .reset_index()
)
summary_std = (
    raw_results
    .groupby(["test_group", "condition", "condition_label", "Controller"], dropna=False)[metric_columns]
    .std()
    .reset_index()
)
summary_std = summary_std.rename(columns={col: f"{col}_std" for col in metric_columns})
summary_results = summary_mean.merge(
    summary_std,
    on=["test_group", "condition", "condition_label", "Controller"],
    how="left",
)

summary_path = RESULT_TABLE_DIR / "robustness_summary_by_condition.csv"
summary_results.to_csv(summary_path, index=False)

controller_summary_path = RESULT_TABLE_DIR / "robustness_controller_summary.csv"
controller_summary = (
    summary_results[summary_results["test_group"] != "nominal"]
    .groupby("Controller")
    .agg(
        mean_degradation_percent=("performance_degradation_percent", "mean"),
        worst_degradation_percent=("performance_degradation_percent", "max"),
        mean_J_total=("J_total", "mean"),
        worst_J_total=("J_total", "max"),
        mean_voltage_max_abs=("voltage_max_abs", "mean"),
        max_saturation_percent=("saturation_percent", "max"),
    )
    .reset_index()
)
controller_summary.to_csv(controller_summary_path, index=False)

print(f"Saved: {summary_path.relative_to(PROJECT_ROOT)}")
print(f"Saved: {controller_summary_path.relative_to(PROJECT_ROOT)}")
display(summary_results.round(6))
display(controller_summary.round(6))


Saved: results/tables/robustness_summary_by_condition.csv
Saved: results/tables/robustness_controller_summary.csv


,test_group,condition,condition_label,Controller,J_total,J_tracking,J_overshoot,J_control,overshoot_percent,settling_time,...,J_overshoot_std,J_control_std,overshoot_percent_std,settling_time_std,steady_state_error_percent_std,voltage_max_abs_std,saturation_percent_std,omega_final_std,disturbance_recovery_time_std,performance_degradation_percent_std
0,inertia,J_0.8x,J_m = 0.8 J0,Manual PID,0.037745,0.000109,0.000000,0.251200,0.000000,1.4550,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,inertia,J_0.8x,J_m = 0.8 J0,Optimized PID,0.036321,0.000570,0.000000,0.239863,0.000000,1.7800,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,inertia,J_0.8x,J_m = 0.8 J0,RL Direct Voltage Control,0.034642,0.007762,0.000000,0.199894,0.000000,inf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,inertia,J_1.0x,J_m = J0,Manual PID,0.038177,0.000114,0.000000,0.254054,0.000000,1.3220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,inertia,J_1.0x,J_m = J0,Optimized PID,0.036702,0.000611,0.000000,0.242238,0.008660,1.5470,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,inertia,J_1.0x,J_m = J0,RL Direct Voltage Control,0.034903,0.007779,0.000000,0.201571,0.000000,inf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,inertia,J_1.2x,J_m = 1.2 J0,Manual PID,0.038627,0.000123,0.000000,0.257021,0.000000,1.1400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,inertia,J_1.2x,J_m = 1.2 J0,Optimized PID,0.037117,0.000666,0.000038,0.244719,0.614055,1.3630,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,inertia,J_1.2x,J_m = 1.2 J0,RL Direct Voltage Control,0.035165,0.007800,0.000000,0.203231,0.000000,inf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,load_disturbance,load_step_0.1569_Nm,"T_L = 0 before 5 s, then 0.1569 N m",Manual PID,0.047874,0.000260,0.000000,0.318120,0.000000,5.9820,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Controller,mean_degradation_percent,worst_degradation_percent,mean_J_total,worst_J_total,mean_voltage_max_abs,max_saturation_percent
0,Manual PID,3.209866,40.221658,0.039402,0.053532,10.252182,0.000000
1,Optimized PID,3.200048,39.698524,0.037877,0.051272,6.164834,0.000000
2,RL Direct Voltage Control,3.095128,33.937739,0.035984,0.046749,12.000000,0.649935


## 주요 응답 그래프

아래 figure는 condition별 speed response와 control voltage를 저장한다. Sensor noise는 첫 번째 seed를 representative trace로 사용하고, 표에서는 5개 seed 평균과 표준편차를 사용한다.


In [7]:
def save_response_plot(test_group, conditions, filename_suffix, title_prefix):
    fig, axes = plt.subplots(len(conditions), 2, figsize=(13, 4.0 * len(conditions)), squeeze=False)

    for row_idx, condition in enumerate(conditions):
        condition_name = condition["condition"]
        condition_label = condition["description"]
        ax_speed, ax_voltage = axes[row_idx]

        for controller in controllers:
            name = controller["name"]
            result = representative_results[(name, test_group, condition_name)]
            ax_speed.plot(result["time"], result["omega"], label=name)
            ax_voltage.plot(result["time"], result["voltage"], label=name)

        ax_speed.axhline(omega_ref, color="black", linestyle="--", linewidth=1, label="Reference")
        ax_speed.axhline(omega_ref * 1.02, color="gray", linestyle=":", linewidth=1)
        ax_speed.axhline(omega_ref * 0.98, color="gray", linestyle=":", linewidth=1)
        if test_group == "load_disturbance":
            ax_speed.axvline(LOAD_STEP_TIME, color="tab:red", linestyle="--", linewidth=1)
            ax_voltage.axvline(LOAD_STEP_TIME, color="tab:red", linestyle="--", linewidth=1)

        ax_voltage.axhline(V_max, color="black", linestyle="--", linewidth=1)
        ax_voltage.axhline(-V_max, color="black", linestyle="--", linewidth=1)

        ax_speed.set_title(f"{title_prefix}: {condition_label}")
        ax_speed.set_xlabel("Time [s]")
        ax_speed.set_ylabel("Speed [rad/s]")
        ax_voltage.set_title(f"Voltage: {condition_label}")
        ax_voltage.set_xlabel("Time [s]")
        ax_voltage.set_ylabel("Voltage [V]")
        ax_speed.legend(loc="best")
        ax_voltage.legend(loc="best")

    fig.tight_layout()
    path = RESULT_FIGURE_DIR / f"robustness_{filename_suffix}.png"
    fig.savefig(path)
    plt.close(fig)
    return path


figure_paths = []
figure_paths.append(save_response_plot(
    "resistance",
    robustness_conditions["resistance"],
    "resistance_variation",
    "Resistance variation speed",
))
figure_paths.append(save_response_plot(
    "inertia",
    robustness_conditions["inertia"],
    "inertia_variation",
    "Inertia variation speed",
))
figure_paths.append(save_response_plot(
    "load_disturbance",
    robustness_conditions["load_disturbance"],
    "load_disturbance",
    "Load disturbance speed",
))
figure_paths.append(save_response_plot(
    "sensor_noise",
    robustness_conditions["sensor_noise"],
    "sensor_noise",
    "Sensor noise speed",
))

for path in figure_paths:
    print(f"Saved: {path.relative_to(PROJECT_ROOT)}")


Saved: results/figures/robustness_resistance_variation.png
Saved: results/figures/robustness_inertia_variation.png
Saved: results/figures/robustness_load_disturbance.png
Saved: results/figures/robustness_sensor_noise.png


## Nominal 대비 degradation 요약

각 controller의 nominal cost를 기준으로 performance degradation을 계산한다.

$$
\mathrm{Degradation}[\%]
=
\frac{J_{\mathrm{test}}-J_{\mathrm{nominal}}}
{J_{\mathrm{nominal}}}
\times 100
$$

값이 양수이면 nominal보다 cost가 증가한 것이고, 음수이면 해당 조건에서 nominal보다 cost가 낮아진 것이다.


In [8]:
degradation_pivot = summary_results.pivot_table(
    index=["test_group", "condition_label"],
    columns="Controller",
    values="performance_degradation_percent",
    aggfunc="mean",
)

degradation_path = RESULT_TABLE_DIR / "robustness_degradation_summary.csv"
degradation_pivot.reset_index().to_csv(degradation_path, index=False)

print(f"Saved: {degradation_path.relative_to(PROJECT_ROOT)}")
display(degradation_pivot.round(3))

fig, ax = plt.subplots(figsize=(10, 6))
plot_data = (
    summary_results[summary_results["test_group"] != "nominal"]
    .assign(label=lambda df: df["test_group"] + " | " + df["condition_label"])
)
for controller_name, group in plot_data.groupby("Controller"):
    ax.plot(
        group["label"],
        group["performance_degradation_percent"],
        marker="o",
        label=controller_name,
    )
ax.axhline(0.0, color="black", linewidth=1)
ax.set_ylabel("Performance degradation [%]")
ax.set_xlabel("Robustness condition")
ax.set_title("Performance Degradation vs Nominal")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
degradation_figure_path = RESULT_FIGURE_DIR / "robustness_degradation_summary.png"
fig.savefig(degradation_figure_path)
plt.close(fig)
print(f"Saved: {degradation_figure_path.relative_to(PROJECT_ROOT)}")


Saved: results/tables/robustness_degradation_summary.csv


Controller                                            Manual PID  \
test_group       condition_label                                   
inertia          J_m = 0.8 J0                             -1.129   
                 J_m = 1.2 J0                              1.180   
                 J_m = J0                                  0.000   
load_disturbance T_L = 0 before 5 s, then 0.1569 N m      25.402   
nominal          Nominal                                   0.000   
resistance       R = 0.8 R0                              -33.582   
                 R = 1.2 R0                               40.222   
                 R = R0                                    0.000   
sensor_noise     sigma = 0 rad/s                           0.000   
                 sigma = 0.005 rad/s                       0.002   
                 sigma = 0.01 rad/s                        0.005   

Controller                                            Optimized PID  \
test_group       condition_label                                      
inertia          J_m = 0.8 J0                                -1.037   
                 J_m = 1.2 J0                                 1.130   
                 J_m = J0                                     0.000   
load_disturbance T_L = 0 before 5 s, then 0.1569 N m         25.720   
nominal          Nominal                                      0.000   
resistance       R = 0.8 R0                                 -33.515   
                 R = 1.2 R0                                  39.699   
                 R = R0                                       0.000   
sensor_noise     sigma = 0 rad/s                              0.000   
                 sigma = 0.005 rad/s                          0.001   
                 sigma = 0.01 rad/s                           0.003   

Controller                                            RL Direct Voltage Control  
test_group       condition_label                                                 
inertia          J_m = 0.8 J0                                            -0.750  
                 J_m = 1.2 J0                                             0.749  
                 J_m = J0                                                 0.000  
load_disturbance T_L = 0 before 5 s, then 0.1569 N m                     27.137  
nominal          Nominal                                                  0.000  
resistance       R = 0.8 R0                                             -30.124  
                 R = 1.2 R0                                              33.938  
                 R = R0                                                   0.000  
sensor_noise     sigma = 0 rad/s                                          0.000  
                 sigma = 0.005 rad/s                                      0.000  
                 sigma = 0.01 rad/s                                       0.002

Saved: results/figures/robustness_degradation_summary.png


## 해석 / 주의사항

아래 cell은 저장된 결과를 기준으로 간단한 해석 문장을 만든다. 더 자세한 결론은 논문/보고서 작성 단계에서 control objective와 feasibility 기준을 함께 고려해 정리한다.


In [9]:
best_nominal = (
    summary_results[summary_results["test_group"] == "nominal"]
    .sort_values("J_total")
    .iloc[0]
)
worst_degradation = (
    summary_results[summary_results["test_group"] != "nominal"]
    .sort_values("performance_degradation_percent", ascending=False)
    .iloc[0]
)
load_recovery = summary_results[summary_results["test_group"] == "load_disturbance"][
    ["Controller", "J_total", "performance_degradation_percent", "disturbance_recovery_time"]
].sort_values("disturbance_recovery_time")

print("해석 요약")
print(f"- Nominal J_total 기준 최저 cost controller: {best_nominal['Controller']} (J_total={best_nominal['J_total']:.6f})")
print(
    "- 가장 큰 nominal 대비 degradation: "
    f"{worst_degradation['Controller']} / {worst_degradation['condition_label']} "
    f"({worst_degradation['performance_degradation_percent']:.2f}%)"
)
print("- Load disturbance recovery time:")
display(load_recovery.round(6))

print("\n주의사항")
print("- RL Direct Voltage Control은 best deterministic evaluation checkpoint를 그대로 읽었으며, 이 notebook에서는 재학습하지 않았다.")
print("- Load torque 0.1569 N m는 논문에 직접 제시된 절대값이 아니라, 이 프로젝트 nominal friction torque의 25%로 환산한 조건이다.")
print("- Sensor noise는 true motor state가 아니라 controller feedback measurement에만 들어간다.")


해석 요약
- Nominal J_total 기준 최저 cost controller: RL Direct Voltage Control (J_total=0.034903)
- 가장 큰 nominal 대비 degradation: Manual PID / R = 1.2 R0 (40.22%)
- Load disturbance recovery time:


,Controller,J_total,performance_degradation_percent,disturbance_recovery_time
9,Manual PID,0.047874,25.401532,0.982
10,Optimized PID,0.046142,25.719889,1.433
11,RL Direct Voltage Control,0.044375,27.137042,inf



주의사항
- RL Direct Voltage Control은 best deterministic evaluation checkpoint를 그대로 읽었으며, 이 notebook에서는 재학습하지 않았다.
- Load torque 0.1569 N m는 논문에 직접 제시된 절대값이 아니라, 이 프로젝트 nominal friction torque의 25%로 환산한 조건이다.
- Sensor noise는 true motor state가 아니라 controller feedback measurement에만 들어간다.


## 저장 파일 확인


In [10]:
expected_outputs = [
    raw_results_path,
    summary_path,
    controller_summary_path,
    degradation_path,
    *figure_paths,
    degradation_figure_path,
]

missing = [path for path in expected_outputs if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing expected outputs: {missing}")

pd.DataFrame({
    "output": [str(path.relative_to(PROJECT_ROOT)) for path in expected_outputs],
    "size_bytes": [path.stat().st_size for path in expected_outputs],
})


,output,size_bytes
0,results/tables/robustness_all_results.csv,20694
1,results/tables/robustness_summary_by_condition...,9408
2,results/tables/robustness_controller_summary.csv,483
3,results/tables/robustness_degradation_summary.csv,828
4,results/figures/robustness_resistance_variatio...,277610
5,results/figures/robustness_inertia_variation.png,274502
6,results/figures/robustness_load_disturbance.png,111250
7,results/figures/robustness_sensor_noise.png,299376
8,results/figures/robustness_degradation_summary...,172299
